In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
os.environ['CASTLE_BACKEND'] = 'pytorch'
import sys
sys.path.append("../..")
sys.path.append("../../src")
import numpy as np
from castle.algorithms import GES
from src.base.dataloader import Dataset
from src.tools.metric import get_compared_components, metric_skeleton_level, metric_target_level, metric_cpdag_level

### exp_1——hard+single

In [ ]:
exp_name = 'exp_200'
intervention_size_list = [1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../../baselines/exps_of_result/ges/{exp_name}/raw', exist_ok=True)
os.makedirs(f'../../baselines/exps_of_result/ges/{exp_name}/prune', exist_ok=True)

for idx, benchmark_name in enumerate(['01earthquake', '02survey', '03asia', '04sachs',  '05child', '06insurance', '07water', '08mildew', '09alarm', '10barley', '11hailfinder', '12hepar2', '13win95pts', '14pathfinder']):
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    aug_dataset_path = f'../../datasets/experiment/original/{exp_name}/aug_samples/{benchmark_name}_aug_dataset.npy'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    # try:
    raw_pred_graph_path = f'../../baselines/exps_of_result/ges/{exp_name}/raw/{benchmark_name}_aug_graph.txt'
    prune_pred_graph_path = f'../../baselines/exps_of_result/ges/{exp_name}/prune/{benchmark_name}_aug_graph.txt'
    
    ges = GES()
    dataset = Dataset(data_path=aug_dataset_path).to_pandas()
    ges.learn(dataset)

    pred_adj = ges.causal_matrix
    with open(raw_pred_graph_path, 'wb') as f:
        np.savetxt(f, pred_adj, fmt='%i')
    
    pred_adj[:,-intervention_size:] = 0  # prune all sys->env /  env->env
    with open(prune_pred_graph_path, 'wb') as f:
        np.savetxt(f, pred_adj, fmt='%i')
    
    pred_I_SKELETON, pred_I_TARGETS, pred_I_CPDAG = get_compared_components(raw_pred_graph_path, intervention_size, real=False)
    mt_skeleton = metric_skeleton_level(pred_I_SKELETON, targ_I_SKELETON)
    mt_target = metric_target_level(pred_I_TARGETS, targ_I_TARGETS)
    mt_cpdag = metric_cpdag_level(pred_I_CPDAG, targ_I_CPDAG)
    print(f'raw performance: \n skeleton:{mt_skeleton} \n targets:{mt_target} \n cpdag:{mt_cpdag} \n\n')

    pred_I_SKELETON, pred_I_TARGETS, pred_I_CPDAG = get_compared_components(prune_pred_graph_path, intervention_size, real=False)
    mt_skeleton = metric_skeleton_level(pred_I_SKELETON, targ_I_SKELETON)
    mt_target = metric_target_level(pred_I_TARGETS, targ_I_TARGETS)
    mt_cpdag = metric_cpdag_level(pred_I_CPDAG, targ_I_CPDAG)
    print(f'prune performance: \n skeleton:{mt_skeleton} \n targets:{mt_target} \n cpdag:{mt_cpdag} \n\n')
    # except:
    #     print(f'pass {benchmark_name}\n')

### exp_2——hard+multiple

In [ ]:
exp_name = 'exp_2'
intervention_size_list = [2, 2, 2, 5, 5, 5, 5, 5, 5, 10, 10, 10, 20, 20, 20]
os.makedirs(f'../../baselines/exps_of_result/ges/{exp_name}/raw', exist_ok=True)
os.makedirs(f'../../baselines/exps_of_result/ges/{exp_name}/prune', exist_ok=True)

for idx, benchmark_name in enumerate(['survey', 'asia', 'sachs',  'child', 'insurance', 'water', 'mildew', 'alarm', 'barley', 'hailfinder', 'hepar2', 'win95pts', 'pathfinder', 'munin1', 'andes']):
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    aug_dataset_path = f'../../datasets/experiment/original/{exp_name}/aug_samples/{benchmark_name}_aug_dataset.npy'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size)
    
    try:
        raw_pred_graph_path = f'../../baselines/exps_of_result/ges/{exp_name}/raw/{benchmark_name}_aug_graph.txt'
        prune_pred_graph_path = f'../../baselines/exps_of_result/ges/{exp_name}/prune/{benchmark_name}_aug_graph.txt'
        
        ges = GES()
        dataset = Dataset(data_path=aug_dataset_path).to_pandas()
        ges.learn(dataset)

        pred_adj = ges.causal_matrix
        with open(raw_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_adj, fmt='%i')
        
        pred_adj[:,-intervention_size:] = 0  # prune all sys->env /  env->env
        with open(prune_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_adj, fmt='%i')
        
        pred_I_SKELETON, pred_I_TARGETS, pred_I_CPDAG = get_compared_components(raw_pred_graph_path, intervention_size)
        mt_skeleton = metric_skeleton_level(pred_I_SKELETON, targ_I_SKELETON)
        mt_target = metric_target_level(pred_I_TARGETS, targ_I_TARGETS)
        mt_cpdag = metric_cpdag_level(pred_I_CPDAG, targ_I_CPDAG)
        print(f'raw performance: \n skeleton:{mt_skeleton} \n targets:{mt_target} \n cpdag:{mt_cpdag} \n\n')

        pred_I_SKELETON, pred_I_TARGETS, pred_I_CPDAG = get_compared_components(prune_pred_graph_path, intervention_size)
        mt_skeleton = metric_skeleton_level(pred_I_SKELETON, targ_I_SKELETON)
        mt_target = metric_target_level(pred_I_TARGETS, targ_I_TARGETS)
        mt_cpdag = metric_cpdag_level(pred_I_CPDAG, targ_I_CPDAG)
        print(f'prune performance: \n skeleton:{mt_skeleton} \n targets:{mt_target} \n cpdag:{mt_cpdag} \n\n')
    except:
        print(f'pass {benchmark_name}\n')